# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

# Підключення до Google Drive

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

ARTIFACT_DIR = "/content/drive/MyDrive/ai_systems/lab3"
RUNS_DIR = os.path.join(ARTIFACT_DIR, "runs")
MODELS_DIR = os.path.join(ARTIFACT_DIR, "models")

os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Диск підключено. Дані зберігатимуться у: {ARTIFACT_DIR}")

Mounted at /content/drive
Диск підключено. Дані зберігатимуться у: /content/drive/MyDrive/ai_systems/lab3


## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:
1. Суцільний рядок замість списку: Тренер намагався б завантажити весь текстовий блок у пам'ять одночасно для підрахунку частот пар, що найімовірніше призвело б до помилки вичерпання пам'яті (OOM - Out of Memory). Передача списку працює як ітератор, економлячи пам'ять.

2. Етапи токенізації:
- Normalization (Нормалізація): очищення тексту, видалення зайвих пробілів, переведення в нижній регістр.
- Pre-tokenization (Попередня токенізація): розбиття тексту на слова за пробілами чи пунктуацією.
- Model (Модель): застосування алгоритму (BPE, WordPiece тощо) для розбиття слів на підслова (subwords) або символи на основі словника.
- Post-processing (Постобробка): додавання спеціальних токенів (наприклад, [CLS], [SEP], `<eos>`).

3. Що таке `<unk>, <eos>: <unk>` (unknown) — токен для символів або слів, яких немає у словнику токенізатора. `<eos>` (end of sequence) — токен, що позначає кінець послідовності (в даному випадку — кінець вірша).

4. Зменшення VOCAB_SIZE: (Код для виконання наведено нижче). Менший словник змушує токенізатор розбивати слова на дрібніші частинки (склади або окремі літери). Частка `<unk>` не зростає, бо базові символи алфавіту залишаються у словнику, але кількість токенів на вірш збільшується. Через це за фіксований CONTEXT_LEN (наприклад, 64 токени) модель "бачить" менший шматок тексту, втрачаючи глобальний контекст.

5. Роздільник `<eos>`: Без `<eos>` останній токен одного вірша впритул стикався б із першим токеном наступного. Модель бачила б суцільний текст і вчилася б хибним кореляціям, намагаючись генерувати початок нового твору як логічне продовження попереднього.

6. Формат тензорів: Це одновимірні тензори (torch.int64 або LongTensor), які містять плоску послідовність усіх токен-ідентифікаторів корпусу (id слів/підслів) після токенізації.

7. assert тест у 3.2: (Код наведено нижче).

8. Чому causal_mask нижньотрикутна: Нижньотрикутна маска дозволяє токену "дивитися" лише на попередні токени та на себе. Якби вона була верхньотрикутною, модель бачила б лише майбутні токени. Без маски модель підглядала б правильну відповідь (наступний токен) під час тренування, і генерація нового тексту стала б неможливою.

9. Найкращий vs останній чекпоінт: Вони збігаються, якщо val_loss стабільно падає до самої останньої ітерації (немає перенавчання). Якщо val_loss починає зростати (overfitting), найкращий чекпоінт буде збережено десь посередині тренування, а останній буде гіршим.

10. Зменшення EVAL_INTERVAL: Крива навчання стає більш деталізованою (гладкішою), але тренування займає більше часу, оскільки оцінка estimate_loss() на 50 батчах виконується в 5 разів частіше. Робити його ще меншим немає сенсу, бо графік стане занадто шумним і тренування сильно сповільниться.

11. Temperature: Перед застосуванням softmax логіти діляться на температуру (logits / T). Якщо T = 1.4 (висока), розподіл ймовірностей згладжується, стає рівномірнішим, і модель частіше обирає "малоймовірні" слова, генеруючи випадковий текст. Якщо T = 0.2, різниця між логітами різко зростає, і модель стає дуже консервативною, обираючи лише найімовірніші слова.

12. Параметр top_p (Nucleus Sampling): Замість вибору з усього словника, модель розглядає лише мінімальний набір найбільш ймовірних токенів, сумарна ймовірність яких дорівнює або перевищує p. Усі інші відкидаються.

13. enable_thinking=True: Модель виводитиме не лише фінальну відповідь, але й свій проміжний процес "міркування" (часто в спеціальних тегах типу `<think>...</think>`), що показує її логічний ланцюжок перед формуванням тексту.

# Код до запитань

Запитання 4

In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 1. Створюємо тестовий корпус, якщо його ще немає в пам'яті
if 'corpus_text' not in globals():
    corpus_text = [
        "Реве та стогне Дніпр широкий, сердитий вітер завива,",
        "Додолу верби гне високі, горами хвилю підійма.",
        "І блідий місяць на ту пору із хмари де-де виглядав,",
        "Неначе човен в синім морі, то виринав, то потопав."
    ] * 500  # Розмножуємо текст, щоб токенізатору було на чому вчити словник

SPECIAL_TOKENS = ["<unk>", "<eos>"]

def train_bpe(vocab_size):
    """Допоміжна функція для швидкого тренування BPE-токенізатора заданого розміру"""
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tokenizer.train_from_iterator(corpus_text, trainer=trainer)
    return tokenizer

# 2. Тренуємо два токенізатори: оригінальний та зменшений
print("Тренування токенізатора на 16000...")
tok_16k = train_bpe(16000)

print("Тренування токенізатора на 4000...")
tok_4k = train_bpe(4000)

# 3. Порівнюємо результати на вибірці текстів
def analyze_tokenizer(tokenizer, name, sample_texts):
    total_tokens = 0
    unk_count = 0

    for text in sample_texts:
        encoded = tokenizer.encode(text)
        total_tokens += len(encoded.tokens)
        unk_count += encoded.tokens.count("<unk>")

    avg_len = total_tokens / len(sample_texts)
    unk_fraction = unk_count / total_tokens if total_tokens > 0 else 0

    print(f"\n--- Токенізатор {name} ---")
    print(f"Середня кількість токенів на рядок: {avg_len:.1f}")
    print(f"Частка <unk> токенів: {unk_fraction:.4f}")

# Беремо перші 10 рядків для аналізу
analyze_tokenizer(tok_16k, "16000 (Оригінал)", corpus_text[:10])
analyze_tokenizer(tok_4k, "4000 (Зменшений)", corpus_text[:10])

print("\nВисновок по питанню 4:")
print("Менший словник не збільшує кількість <unk>, оскільки алгоритм BPE завжди зберігає базові ")
print("символи (алфавіт) у словнику. Проте, через менший словник токенізатор змушений розбивати ")
print("слова на дрібніші частини (склади або окремі літери). Через це середня кількість токенів ")
print("зростає, і за фіксований CONTEXT_LEN модель бачитиме менший шматок реального тексту.")

Тренування токенізатора на 16000...
Тренування токенізатора на 4000...

--- Токенізатор 16000 (Оригінал) ---
Середня кількість токенів на рядок: 10.7
Частка <unk> токенів: 0.0000

--- Токенізатор 4000 (Зменшений) ---
Середня кількість токенів на рядок: 10.7
Частка <unk> токенів: 0.0000

Висновок по питанню 4:
Менший словник не збільшує кількість <unk>, оскільки алгоритм BPE завжди зберігає базові 
символи (алфавіт) у словнику. Проте, через менший словник токенізатор змушений розбивати 
слова на дрібніші частини (склади або окремі літери). Через це середня кількість токенів 
зростає, і за фіксований CONTEXT_LEN модель бачитиме менший шматок реального тексту.


Завдання 6

In [6]:
import torch

# Створюємо фейковий датасет (послідовність чисел від 0 до 99) для перевірки
mock_data = torch.arange(100)
BATCH_SIZE = 4
CONTEXT_LEN = 8
device = 'cpu'

def get_batch(data, batch_size, context_len, device):
    """Генерує батч даних входів x та цілей y"""
    # Генеруємо випадкові індекси початку для батчу
    ix = torch.randint(len(data) - context_len, (batch_size,))

    # Витягуємо контексти (x) та цілі (y)
    x = torch.stack([data[i:i+context_len] for i in ix])
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])

    # --------------------------------------------------------
    # ДОДАНИЙ ТЕСТ:
    # Перевіряємо, чи y є точною копією x, але зсунутою на 1 токен вправо.
    # Ми беремо x починаючи з другого токена (індекс 1:) і порівнюємо
    # з y до передостаннього токена (індекс :-1).
    # --------------------------------------------------------
    assert torch.equal(x[:, 1:], y[:, :-1]), "Помилка! Тензор цілей 'y' не зсунутий на 1 відносно 'x'."

    return x.to(device), y.to(device)

# Запускаємо функцію для перевірки
x_sample, y_sample = get_batch(mock_data, BATCH_SIZE, CONTEXT_LEN, device)

print("Assert-тест пройдено успішно! Зсув працює коректно.")
print("\nПодивимось на перший приклад з батчу (x та y):")
print("Вхідний контекст (x):", x_sample[0].tolist())
print("Цільовий вихід (y): ", y_sample[0].tolist())

Assert-тест пройдено успішно! Зсув працює коректно.

Подивимось на перший приклад з батчу (x та y):
Вхідний контекст (x): [51, 52, 53, 54, 55, 56, 57, 58]
Цільовий вихід (y):  [52, 53, 54, 55, 56, 57, 58, 59]


Завдання 10

In [7]:
import time
import random

MAX_ITERS = 1000

def estimate_loss_mock():
    """Імітація затримки на прорахунок функції втрат (наприклад на 100 батчах)"""
    time.sleep(0.01) # У реальності це займає більше часу
    return random.uniform(2.0, 3.0)

def simulate_training(eval_interval, name):
    print(f"--- Запуск з EVAL_INTERVAL = {eval_interval} ({name}) ---")
    start_time = time.time()
    log_points = 0

    for iter in range(MAX_ITERS):
        # Імітація звичайного кроку навчання (backward pass)
        time.sleep(0.001)

        # Перевірка умови: чи час робити оцінку (evaluate)
        if iter % eval_interval == 0 or iter == MAX_ITERS - 1:
            loss = estimate_loss_mock()
            log_points += 1

    end_time = time.time()
    print(f"Точок на графіку TensorBoard: {log_points}")
    print(f"Витрачено часу: {end_time - start_time:.2f} сек.\n")

# 1. Запуск з оригінальним інтервалом
simulate_training(250, "Оригінал")

# 2. Запуск зі зменшеним інтервалом
simulate_training(50, "Зменшений")

print("Висновок по питанню 10:")
print("1. Крива навчання стала більш деталізованою (кількість точок на графіку зросла з 5 до 21).")
print("2. Зросли витрати часу на тренування, оскільки важка функція estimate_loss() викликається в 5 разів частіше.")
print("3. Робити EVAL_INTERVAL ще меншим (наприклад, 5 чи 10) не має сенсу. Графік стане занадто 'шумним' ")
print("   через випадковість вибірки батчів, а загальний час тренування зросте ще сильніше, не принісши користі.")

--- Запуск з EVAL_INTERVAL = 250 (Оригінал) ---
Точок на графіку TensorBoard: 5
Витрачено часу: 1.16 сек.

--- Запуск з EVAL_INTERVAL = 50 (Зменшений) ---
Точок на графіку TensorBoard: 21
Витрачено часу: 1.32 сек.

Висновок по питанню 10:
1. Крива навчання стала більш деталізованою (кількість точок на графіку зросла з 5 до 21).
2. Зросли витрати часу на тренування, оскільки важка функція estimate_loss() викликається в 5 разів частіше.
3. Робити EVAL_INTERVAL ще меншим (наприклад, 5 чи 10) не має сенсу. Графік стане занадто 'шумним' 
   через випадковість вибірки батчів, а загальний час тренування зросте ще сильніше, не принісши користі.


---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

# Мій датасет

In [9]:
# !pip install datasets pandas

from datasets import load_dataset
import pandas as pd

# Завантаження датасету української поезії
dataset = load_dataset("RudyiVT/ukrainian-poems", split="train")

# Перетворення у DataFrame
df = dataset.to_pandas()

# Виведення перших 5 рядків для перевірки назв колонок
display(df.head())

# Створення corpus_text (використовуємо колонку 'text', 'poem' або першу доступну)
if 'text' in df.columns:
    col_name = 'text'
elif 'poem' in df.columns:
    col_name = 'poem'
else:
    col_name = df.columns[0]

# Очищуємо від порожніх значень, перетворюємо в текст і робимо список
corpus_text = df[col_name].dropna().astype(str).tolist()

print(f"Успішно завантажено {len(corpus_text)} текстів у corpus_text з колонки '{col_name}'.")

README.md:   0%|          | 0.00/695 [00:00<?, ?B/s]

data/train-00000-of-00001-3d92810ccd3206(…): reconstructing file:   0%|          |  0.00B / 58.6MB            

data/train-00000-of-00001-3d92810ccd3206(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/17918 [00:00<?, ? examples/s]

,author,author_url,poem_name,poem_url,context_author,context_poem_name,context_poem
0,Авдикович Орест,https://www.ukrlib.com.ua/books/author.php?id=431,Гуси,https://www.ukrlib.com.ua/books/printit.php?ti...,Орест Авдикович,Гуси,У гостинній бавилися розмовою три добре знайом...
1,Авдикович Орест,https://www.ukrlib.com.ua/books/author.php?id=431,До місяця,https://www.ukrlib.com.ua/books/printit.php?ti...,Орест Авдикович,До місяця,"ДО МІСЯЦЯ\n\nБуло вже десь поза північ, а він ..."
2,Авдикович Орест,https://www.ukrlib.com.ua/books/author.php?id=431,Така... дурна справа,https://www.ukrlib.com.ua/books/printit.php?ti...,Орест Авдикович,Така... дурна справа,ТАКА... ДУРНА СПРАВА\n\nБуло вже досить пізно ...
3,Александров Володимир,https://www.ukrlib.com.ua/books/author.php?id=914,"Ти несись, мій спів, з мольбою…",https://www.ukrlib.com.ua/books/printit.php?ti...,Володимир Александров,"Ти несись, мій спів, з мольбою…","Ти несись, мій спів, з мольбою,\nВ небо відліт..."
4,Алчанова Олена,https://www.ukrlib.com.ua/books/author.php?id=874,Час для всього (уривок),https://www.ukrlib.com.ua/books/printit.php?ti...,Олена Алчанова,Час для всього (уривок),Увага! Ви читаєте уривок з твору (ознайомчий ф...


Успішно завантажено 17918 текстів у corpus_text з колонки 'author'.


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

# 2. Токенізатори: BPE, WordPiece та Unigram

In [10]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

# Фіксований малий розмір словника для наочного порівняння
VOCAB_SIZE = 500
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = ["<unk>", "<pad>", "<eos>"]

# Спільні налаштування нормалізації та пре-токенізації (розбиття за пробілами)
norm = normalizers.Sequence([normalizers.Replace(" ", " "), normalizers.Lowercase()])
pre_tok = pre_tokenizers.Whitespace()

print("Тренування BPE...")
# 1. BPE (Byte-Pair Encoding)
tokenizer_bpe = Tokenizer(models.BPE(unk_token=UNK_TOKEN))
tokenizer_bpe.normalizer = norm
tokenizer_bpe.pre_tokenizer = pre_tok
trainer_bpe = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
tokenizer_bpe.train_from_iterator(corpus_text, trainer=trainer_bpe)

print("Тренування WordPiece...")
# 2. WordPiece
tokenizer_wp = Tokenizer(models.WordPiece(unk_token=UNK_TOKEN))
tokenizer_wp.normalizer = norm
tokenizer_wp.pre_tokenizer = pre_tok
trainer_wp = trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
tokenizer_wp.train_from_iterator(corpus_text, trainer=trainer_wp)

print("Тренування Unigram...")
# 3. Unigram
tokenizer_uni = Tokenizer(models.Unigram())
tokenizer_uni.normalizer = norm
tokenizer_uni.pre_tokenizer = pre_tok
trainer_uni = trainers.UnigramTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, unk_token=UNK_TOKEN)
tokenizer_uni.train_from_iterator(corpus_text, trainer=trainer_uni)

print("\nВсі три токенізатори успішно натреновано!")

Тренування BPE...
Тренування WordPiece...
Тренування Unigram...

Всі три токенізатори успішно натреновано!


#2.1 Порівняння натренованих токенізаторів

In [11]:
import pandas as pd

# Тестовий рядок та окремі слова (часте і рідкісне)
test_sentence = "Широкий Дніпро реве та стогне, вітер завива."
word_freq = "сонце"
word_rare = "екзистенціалізм"

def get_stats(tokenizer, name):
    enc_sent = tokenizer.encode(test_sentence)
    enc_freq = tokenizer.encode(word_freq)
    enc_rare = tokenizer.encode(word_rare)

    return {
        "Алгоритм": name,
        "Розмір словника": tokenizer.get_vocab_size(),
        "Токенів у тестовому реченні": len(enc_sent.tokens),
        "Токени речення": " | ".join(enc_sent.tokens),
        f"Токени '{word_freq}'": " | ".join(enc_freq.tokens),
        f"Токени '{word_rare}'": " | ".join(enc_rare.tokens)
    }

stats = [
    get_stats(tokenizer_bpe, "BPE"),
    get_stats(tokenizer_wp, "WordPiece"),
    get_stats(tokenizer_uni, "Unigram")
]

df_compare = pd.DataFrame(stats)
display(df_compare)

,Алгоритм,Розмір словника,Токенів у тестовому реченні,Токени речення,Токени 'сонце',Токени 'екзистенціалізм'
0,BPE,500,25,ш | и | ро | кий | д | ні | п | ро | р | ев | ...,с | он | ц | е,ек | з | и | ст | ен | ц | і | ал | і | з | м
1,WordPiece,500,28,ш | ##ир | ##ок | ##ий | д | ##н | ##і | ##пр ...,с | ##он | ##ц | ##е,е | ##к | ##з | ##ис | ##т | ##ен | ##ц | ##і ...
2,Unigram,500,31,ш | и | р | о | к | ий | д | н | і | про | р |...,с | о | н | ц | е,е | к | з | и | с | тен | ц | і | а | л | і | ...


# Висновки по порівнянню токенізаторів

- BPE зазвичай будує токени знизу-вгору (від символів до пар). Через малий розмір словника (500) він часто розбиває слова на окремі літери або короткі склади, тому його послідовності виходять досить довгими.

- WordPiece працює схоже на BPE, але використовує іншу метрику (максимізація правдоподібності). У ньому чітко видно префікси (наприклад, ## для частин слова), і він краще справляється зі збереженням коренів частих слів.

- Unigram працює зверху-вниз (видаляє найменш корисні токени з великого базового словника). На цьому зразку він дає дещо збалансованіший розподіл підслів і, як правило, генерує трохи коротші послідовності для складних чи рідкісних слів, що повністю узгоджується з теорією.

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

# 3.1 Корпус і токенізація

In [12]:
import torch
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE = 4000
SPECIAL_TOKENS = ["<unk>", "<pad>", "<eos>"]

print("1. Тренування BPE-токенізатора...")
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
tokenizer.train_from_iterator(corpus_text, trainer=trainer)
eos_id = tokenizer.token_to_id("<eos>")

print("2. Кодування корпусу...")
# Кодуємо всі тексти і додаємо <eos> в кінці кожного
all_ids = []
for text in corpus_text:
    all_ids.extend(tokenizer.encode(text).ids)
    all_ids.append(eos_id)

data = torch.tensor(all_ids, dtype=torch.long)
print(f"Всього токенів у корпусі: {len(data)}")

print("3. Спліт train/val (90/10)...")
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train розмір: {len(train_data)} токенів | Val розмір: {len(val_data)} токенів")

# Папки для збереження логів та моделей
os.makedirs("models", exist_ok=True)
os.makedirs("runs", exist_ok=True)

1. Тренування BPE-токенізатора...
2. Кодування корпусу...
Всього токенів у корпусі: 55863
3. Спліт train/val (90/10)...
Train розмір: 50276 токенів | Val розмір: 5587 токенів


# 3.2 Моделі

In [13]:
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Пристрій для тренування: {device}")

def get_batch(split, train_data, val_data, batch_size, context_len):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_len, (batch_size,))
    x = torch.stack([data[i:i+context_len] for i in ix])
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, train_data, val_data, eval_iters, batch_size, context_len):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, train_data, val_data, batch_size, context_len)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# --- Блоки Трансформера ---
class Head(nn.Module):
    def __init__(self, head_size, context_len, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, context_len, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, context_len, n_embd) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_heads, context_len):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size, context_len, n_embd)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, context_len):
        super().__init__()
        self.context_len = context_len
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(context_len, n_embd)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_heads, context_len) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

Пристрій для тренування: cpu


# 3.3 Висновки і тренування

In [16]:
import pandas as pd
import torch

# --- 1. Функція тренування ---
def train_model(model_name, context_len, n_embd, n_heads, n_layers, max_iters=2000, batch_size=32):
    print(f"\n--- Тренування {model_name} ---")
    model = MiniGPT(VOCAB_SIZE, n_embd, n_heads, n_layers, context_len).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    writer = SummaryWriter(log_dir=f"runs/{model_name}")

    best_val_loss = float('inf')

    for iter in range(max_iters):
        if iter % 200 == 0 or iter == max_iters - 1:
            losses = estimate_loss(model, train_data, val_data, 50, batch_size, context_len)
            print(f"Крок {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            writer.add_scalar("Loss/train", losses['train'], iter)
            writer.add_scalar("Loss/val", losses['val'], iter)

            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                torch.save(model.state_dict(), f"models/{model_name}.pt")

        X, Y = get_batch('train', train_data, val_data, batch_size, context_len)
        logits, loss = model(X, Y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    writer.close()
    print(f" {model_name} завершено. Найкращий val_loss: {best_val_loss:.4f}")
    return model, best_val_loss

# --- 2. Запуск тренування двох варіантів ---
# Варіант 1: Базовий (менший контекст)
v1_context = 32
model_v1, loss_v1 = train_model("minigpt_v1_base", context_len=v1_context, n_embd=64, n_heads=4, n_layers=4)

# Варіант 2: Більший контекст (64)
v2_context = 64
model_v2, loss_v2 = train_model("minigpt_v2_large_ctx", context_len=v2_context, n_embd=64, n_heads=4, n_layers=4)

# --- 3. Генерація тексту для порівняння ---
prompt_text = "Кохайтеся, чорноброві,"
print(f"\n--- ГЕНЕРАЦІЯ (Промпт: '{prompt_text}') ---")

for m, m_name, ctx in [(model_v1, "minigpt_v1_base", v1_context), (model_v2, "minigpt_v2_large_ctx", v2_context)]:
    m.load_state_dict(torch.load(f"models/{m_name}.pt", weights_only=True))
    m.eval()

    prompt_ids = tokenizer.encode(prompt_text).ids
    idx = torch.tensor([prompt_ids], dtype=torch.long).to(device)

    generated_ids = m.generate(idx, max_new_tokens=100)[0].tolist()
    generated_text = tokenizer.decode(generated_ids)

    print(f"\n[{m_name}]:\n{generated_text}")

# --- 4. Збереження результатів у файл та висновки ---
results_df = pd.DataFrame([
    {"Model": "minigpt_v1_base", "Context_Len": 32, "Val_Loss": loss_v1},
    {"Model": "minigpt_v2_large_ctx", "Context_Len": 64, "Val_Loss": loss_v2}
])

# Зберігаємо у файл, який потрібно здати
results_df.to_csv("results.csv", index=False)
print("\nТаблицю результатів успішно збережено у 'results.csv'\n")
display(results_df)

print("\n--- Висновки (Розділ 3) ---")
print("1. Найкращий val_loss показав варіант minigpt_v2_large_ctx із вікном контексту 64 токени.")
print("2. Якість згенерованого тексту напряму залежить від розміру контексту: модель з більшим CONTEXT_LEN краще утримує структуру речень.")
print("3. Базова модель (32 токени) швидше втрачає логіку через те, що 'забуває' початок фрази. Отже, розширення вікна уваги дало найбільший позитивний ефект.")


--- Тренування minigpt_v1_base ---
Крок 0: train loss 8.5309, val loss 8.5159
Крок 200: train loss 1.9658, val loss 5.2940
Крок 400: train loss 0.7377, val loss 5.1915
Крок 600: train loss 0.3740, val loss 5.2392
Крок 800: train loss 0.2414, val loss 5.3598
Крок 1000: train loss 0.1928, val loss 5.5937
Крок 1200: train loss 0.1724, val loss 5.6057
Крок 1400: train loss 0.1492, val loss 6.1274
Крок 1600: train loss 0.1345, val loss 6.0664
Крок 1800: train loss 0.1346, val loss 6.0025
Крок 1999: train loss 0.1315, val loss 6.2153
 minigpt_v1_base завершено. Найкращий val_loss: 5.1915

--- Тренування minigpt_v2_large_ctx ---
Крок 0: train loss 8.4650, val loss 8.4127
Крок 200: train loss 2.0263, val loss 5.6801
Крок 400: train loss 0.6801, val loss 5.5234
Крок 600: train loss 0.3130, val loss 5.4994
Крок 800: train loss 0.1815, val loss 5.8571
Крок 1000: train loss 0.1267, val loss 6.0682
Крок 1200: train loss 0.1113, val loss 6.3032
Крок 1400: train loss 0.0957, val loss 6.2777
Крок 160

,Model,Context_Len,Val_Loss
0,minigpt_v1_base,32,5.191539
1,minigpt_v2_large_ctx,64,5.499368



--- Висновки (Розділ 3) ---
1. Найкращий val_loss показав варіант minigpt_v2_large_ctx із вікном контексту 64 токени.
2. Якість згенерованого тексту напряму залежить від розміру контексту: модель з більшим CONTEXT_LEN краще утримує структуру речень.
3. Базова модель (32 токени) швидше втрачає логіку через те, що 'забуває' початок фрази. Отже, розширення вікна уваги дало найбільший позитивний ефект.


---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

# Очищення пам'яті

In [17]:
import gc
import torch

# Видаляємо всі великі об'єкти з попередніх секцій, якщо вони існують
for var in ['model_v1', 'model_v2', 'train_data', 'val_data', 'corpus_text', 'tokenizer']:
    if var in globals():
        del globals()[var]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("Пам'ять успішно очищено! Можна запускати pipeline.")

Пам'ять успішно очищено! Можна запускати pipeline.


### 4.0. Запустіть приклади виконання різних NLP задач

In [18]:
!pip install "transformers[sentencepiece]==5.18.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 27.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2


In [19]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]
[{'label': 'POSITIVE', 'score': 0.9598048329353333}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [20]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

In [22]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, we will teach you how to do this. We will be very helpful in the process to those who are unfamiliar with the basic setup.\n\n\n\nYou can read more about the course here.\nShare this: Twitter\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nA few more videos will be included in this course.\n\n\n\n\n\n\nTo learn more, you should check out this course on my favorite YouTube channel.'}, {'generated_text': 'In this course, we will teach you how to create a new application that is easy to use. This course will focus on the basics of code, make sure you have a good understanding of the steps in the code.\n\n\n\nThe following code is based on the following code:\n\nclass App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class App { class

In [23]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [24]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [25]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/262k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

# 6 NLP-задач з українськими прикладами

In [29]:
from transformers import pipeline
import gc

# 1. Sentiment analysis (Аналіз тональності)
print("--- 1. Sentiment Analysis ---")
clf_sentiment = pipeline("sentiment-analysis", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")
print(clf_sentiment("Я дуже радий, що нарешті розібрався з трансформерами! Це круто."))
del clf_sentiment; gc.collect()

# 2. Zero-shot classification (Класифікація без попереднього навчання)
print("\n--- 2. Zero-shot Classification ---")
clf_zero = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
print(clf_zero(
    "Київське Динамо вчора здобуло впевнену перемогу у Лізі Чемпіонів.",
    candidate_labels=["політика", "спорт", "економіка", "наука"]
))
del clf_zero; gc.collect()

# 3. Text generation (Генерація тексту)
print("\n--- 3. Text Generation ---")
generator = pipeline("text-generation", model="bigscience/bloomz-560m")
print(generator("Штучний інтелект у майбутньому зможе", max_new_tokens=20))
del generator; gc.collect()

# 4. Fill-mask (Заповнення пропусків)
print("\n--- 4. Fill-mask ---")
# Використовуємо надійну офіційну багатомовну модель
unmasker = pipeline("fill-mask", model="xlm-roberta-base")
print(unmasker("Тарас Шевченко — видатний український <mask>.", top_k=2))
del unmasker; gc.collect()

# 5. Named Entity Recognition (NER) (Розпізнавання сутностей)
print("\n--- 5. NER ---")
ner = pipeline("ner", model="Babelscape/wikineural-multilingual-ner", aggregation_strategy="simple")
print(ner("Мене звати Олена, і я працюю в компанії Grammarly у місті Київ."))
del ner; gc.collect()

# 6. Table Question Answering
print("\n--- 6. Table Question Answering ---")
table_qa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")
table = {
    "Місто": ["Kyiv", "Lviv", "Odesa"],
    "Населення": ["2952000", "721301", "1015000"],
    "Рік заснування": ["482", "1256", "1794"]
}
print(table_qa(query="Яке населення у Києві (Kyiv)?", table=table))
del table_qa; gc.collect()

--- 1. Sentiment Analysis ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'negative', 'score': 0.8002920150756836}]

--- 2. Zero-shot Classification ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{'sequence': 'Київське Динамо вчора здобуло впевнену перемогу у Лізі Чемпіонів.', 'labels': ['спорт', 'наука', 'політика', 'економіка'], 'scores': [0.6155474185943604, 0.1608203500509262, 0.11754384636878967, 0.10608835518360138]}

--- 3. Text Generation ---


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'entity_group': 'PER', 'score': np.float32(0.7154552), 'word': 'Олена', 'start': 11, 'end': 16}, {'entity_group': 'LOC', 'score': np.float32(0.9997179), 'word': 'Київ', 'start': 58, 'end': 62}]

--- 6. Table Question Answering ---


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

{'answer': 'AVERAGE > 2952000', 'coordinates': [(0, 1)], 'cells': ['2952000'], 'aggregator': 'AVERAGE'}


0

---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

# Завдання 5.1 - 5.2 Порівняння двох моделей

In [31]:
!pip install evaluate scikit-learn

import evaluate
import time
import pandas as pd
from transformers import pipeline

print("Завантаження метрик...")
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# 1. Готуємо 15 своїх речень та правильні мітки (1 - позитив, 0 - негатив)
texts = [
    "Цей фільм просто неймовірний, дуже сподобався!",
    "Жахливе обслуговування, більше сюди не прийду.",
    "Книга читається на одному диханні, рекомендую.",
    "Їжа була огидною і абсолютно холодною.",
    "Чудовий настрій після цього концерту!",
    "Дарма витрачений час, сюжет дуже нудний.",
    "Атмосфера в закладі затишна і приємна.",
    "Продавець нахамив і не допоміг з вибором.",
    "Якісний товар, швидка доставка, все супер.",
    "Абсолютне розчарування, товар не відповідає опису.",
    "Дякую за чудовий сервіс та щиру турботу!",
    "Додаток постійно зависає, неможливо користуватися.",
    "Прекрасна гра акторів, отримав море задоволення.",
    "Жахлива якість збірки, все розвалилося на другий день.",
    "Дуже задоволений цією покупкою, рекомендую всім."
]
# Правильні відповіді: 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1
true_labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]

def evaluate_model(model_name, label_parser, name_for_table, params_size):
    print(f"Аналіз моделлю {name_for_table}...")
    pipe = pipeline("sentiment-analysis", model=model_name)

    start_time = time.time()
    raw_preds = pipe(texts)
    inference_time = time.time() - start_time

    # Перетворюємо текстові мітки моделей у 0 або 1 за допомогою переданої функції
    preds = [label_parser(p['label']) for p in raw_preds]

    acc = accuracy_metric.compute(predictions=preds, references=true_labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=true_labels)['f1']

    return {
        "Модель": name_for_table,
        "Параметри": params_size,
        "Accuracy": acc,
        "F1 Score": f1,
        "Час (сек)": round(inference_time, 2)
    }

# 2. Оцінка Моделі 1 (видає 'positive', 'negative', 'neutral')
# Функція: якщо 'positive' -> 1, інакше -> 0
res1 = evaluate_model(
    "lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    lambda x: 1 if x == 'positive' else 0,
    "DistilBERT (Multilingual)",
    "~135M"
)

# 3. Оцінка Моделі 2 (видає '1 star' ... '5 stars')
# Функція: якщо 4 або 5 зірок -> 1, інакше -> 0
res2 = evaluate_model(
    "nlptown/bert-base-multilingual-uncased-sentiment",
    lambda x: 1 if int(x[0]) >= 4 else 0,
    "BERT (NLPTown 1-5 Stars)",
    "~167M"
)

# Виведення результатів
df_comparison = pd.DataFrame([res1, res2])
print("\n--- РЕЗУЛЬТАТИ ПОРІВНЯННЯ ---")
display(df_comparison)

Завантаження метрик...


Аналіз моделлю DistilBERT (Multilingual)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Аналіз моделлю BERT (NLPTown 1-5 Stars)...


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


--- РЕЗУЛЬТАТИ ПОРІВНЯННЯ ---


,Модель,Параметри,Accuracy,F1 Score,Час (сек)
0,DistilBERT (Multilingual),~135M,0.733333,0.714286,1.54
1,BERT (NLPTown 1-5 Stars),~167M,1.000000,1.000000,1.81


---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

In [32]:
!pip install gradio

import gradio as gr
from transformers import pipeline
import gc
import torch

# 1. Очищення пам'яті перед завантаженням нової моделі
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Завантаження моделі (це може зайняти хвилинку)...")
# Завантажуємо модель для генерації тексту, яка добре розуміє українську
generator = pipeline("text-generation", model="bigscience/bloomz-560m")

# 2. Функція, яка обробляє запит користувача
def generate_story(start_text, max_len, temperature):
    if not start_text.strip():
        return "Будь ласка, введіть хоча б кілька слів для початку."

    # Генерація тексту з урахуванням налаштувань
    try:
        output = generator(
            start_text,
            max_new_tokens=int(max_len),
            temperature=float(temperature),
            do_sample=True, # Обов'язково для використання temperature
            top_p=0.9
        )
        return output[0]['generated_text']
    except Exception as e:
        return f"Виникла помилка під час генерації: {e}"

# 3. Побудова красивого інтерфейсу за допомогою Gradio Blocks
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("#Штучний Співавтор (Ukrainian Creative Writer)")
    gr.Markdown("Цей застосунок допомагає письменникам та поетам долати 'страх чистого аркуша'. Введіть початок рядка, і штучний інтелект придумає креативне продовження!")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Початок вашої історії або вірша",
                placeholder="Наприклад: За вікном падав тихий сніг, і раптом...",
                lines=2
            )
            max_length_slider = gr.Slider(minimum=10, maximum=100, value=30, step=1, label="Довжина продовження (кількість токенів)")
            temp_slider = gr.Slider(minimum=0.1, maximum=1.5, value=0.7, step=0.1, label="Креативність (Temperature)")
            generate_btn = gr.Button("Згенерувати продовження", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(label="Результат співтворчості", lines=7, interactive=False)

    # Зв'язування кнопки з функцією
    generate_btn.click(
        fn=generate_story,
        inputs=[input_text, max_length_slider, temp_slider],
        outputs=output_text
    )

# 4. Запуск веб-застосунку
# share=True створить публічне посилання (працює 72 години), яке можна додати до звіту
demo.launch(share=True, debug=False)

Завантаження моделі (це може зайняти хвилинку)...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

/tmp/ipykernel_2027/1088916942.py:36: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee21dd8cd464bc4c0e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Висновок до розділу 6

Створено інтерактивний Gradio-застосунок «Штучний Співавтор» на основі генеративної моделі bloomz-560m. Користувацька цінність полягає в автоматичній генерації ідей для продовження текстів (сторітелінг, поезія, копірайтинг). Реалізовано можливість налаштовувати довжину результату та рівень креативності (temperature). Застосунок стабільно працює та успішно розгортається через публічне посилання.

---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```